# Inference Attacks
This notebook sets up the environment, prepares data, and runs inference attacks using the `qwq-32b` model.


**Important Workflow Note:**
* **Initial Run:** You must run `!python src/sampling.py` first to generate the initial data.
* **Resuming:** If you are resuming a previous session, you must run **"4. Prepare Data"** and **"6. Prepare Raw Generation Data"** to restore the required files.

## 1. Environment Setup & Authentication
Initialize secrets and authenticate with Hugging Face to access required models/datasets.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Initialize the secrets client
user_secrets = UserSecretsClient()

# 2. Retrieve your HF_TOKEN secret securely
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    
    # 3. Set it as an environment variable for Hugging Face libraries
    os.environ["HF_TOKEN"] = hf_token
    
    # 4. Authenticate the session
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face!")
    
except Exception as e:
    print(f"Error loading HF_TOKEN from Secrets: {e}")
    print("Make sure you added the secret with the exact label 'HF_TOKEN' under Add-ons > Secrets.")

## 2. Clone Repository
Clone the specific branch `Apply_Attacks` of the target repository and navigate into the project directory.

In [ ]:
# Clone the repository containing the attack logic
!git clone -b Apply_Attacks https://github.com/bharathgaddam1712/cot-compliance-safety.git
# Change the current working directory to the cloned repository
%cd cot-compliance-safety

## 3. Install Dependencies
Install the required Python packages silently to keep the output clean.

In [ ]:
# Install required dependencies from requirements.txt
!pip -q install -r requirements.txt

## 4. Prepare Data

### Run Sampling (Initial Run Only)
This script must be executed first to generate the initial dataset.

In [ ]:
# Run the sampling script (Required on first run)
!python src/sampling.py

Creates necessary directories and copy the required dataset (`job_list.jsonl`) from inputs.

In [ ]:
# # Create the internal 'data' directory required by the configuration
# !mkdir -p data

# # Copy the job_list.jsonl file from the read-only input dataset to our working directory
# !cp "ENTER_PATH" ./data/

# # Verify that the file has been copied successfully
# !ls data/

Set up the raw data folder with previously generated outputs from inputs.

In [ ]:
# # Create a directory for raw generation results
# !mkdir -p data/raw

# # Copy previously generated outputs from inputs
# !cp "ENTER_PATH" data/raw

# # Verify the raw data files are present
# !ls data/raw

## 5. Validate Data
Perform a quick sanity check on the dataset to ensure jobs are structured correctly and attacks are properly applied.

In [ ]:
import json, collections

# Load all jobs from the dataset
jobs = [json.loads(l) for l in open("data/job_list.jsonl")]

# Count the occurrence of each attack method
by_atk = collections.Counter(j["attack_method"] for j in jobs)
print(f"total jobs : {len(jobs)}")
print(f"by attack  : {dict(by_atk)}")

# Ensure all jobs have the required 'messages' field
missing = [j for j in jobs if not j.get("messages")]
print(f"missing messages field: {len(missing)}  ← must be 0")
assert len(missing) == 0, "STOP — messages field missing, attacks not applied"

# Perform a spot check on specific attack methods
h = next(j for j in jobs if j["attack_method"] == "h_cot")
r = next(j for j in jobs if j["attack_method"] == "race")
print(f"h_cot roles : {[m['role'] for m in h['messages']]}")  # Expected: ['system','user']
print(f"race  roles : {[m['role'] for m in r['messages']]}")  # Expected: ['user','assistant','user']
print("SPOT CHECK PASSED ✓")

## 6. Run Inference
Execute the inference script using the specified model like `qwq-32b`.

In [ ]:
# Start the inference process
!python src/inference.py --model qwq-32b